# JetMoE-8B Domain Specialization — Aggregate Routing Statistics + UMAP

Generates `jetmoe_domain_specialization.json` (aggregate per-domain routing statistics:
`activation_rate`, `avg_prob`, `specialization_score`, `layer_divergence`, `domain_rate`,
`top_specialists`) plus `jetmoe_domain_specialization_umap.json` (2D UMAP projection of the
same per-(layer, expert) activation-rate vectors, one dimension per domain) — the JetMoE
counterpart of `extract_domain_specialization.ipynb`, emitting the **identical schema** so the
frontend's Domain Specialization tab reads it with no per-model special-casing.

**Same corpus as OLMoE.** The 6 domain passages below are copied verbatim from
`extract_domain_specialization.ipynb` (`code`, `math`, `biomedical`, `legal`,
`creative_writing`, `conversational`; one long ~300-400 word passage each). Only the model
differs, so any difference in the resulting statistics is a difference between the models —
not between the texts. Token *counts* still differ, since JetMoE tokenizes the same passage
differently; that's recorded per domain in `token_counts`.

**Model config is identical to `extract_routing_trace_jetmoe.ipynb`** — same `EXPECTED` field
values, same legacy-config-key patching, same transformers-from-source fallback, and the same
**GQA head check on the config**: the model is loaded with **32 query heads over 16 shared K/V heads**
(`JetMoeConfig.__post_init__` derives `num_attention_heads = num_key_value_heads ×
num_experts_per_tok` = 16 × 2 = 32, because each of the top-2 selected attention experts
contributes its own 16 query heads and both read the *same* 16 K/V heads — 2 query heads per
K/V head, paper Table 1 / research doc §2.2). That check runs unconditionally and before the
weights load, and a `kv_proj` weight-shape assert proves the 16 × 128 K/V split structurally
from the checkpoint rather than from a config number. (The routing-trace notebook additionally
asserts each attention expert's Q/O weight shapes; those stay there, since it is the notebook
that actually reads them.) Attention still runs, MoA router
included — it is simply still not *recorded* (next paragraph). What is *not* carried over is
the deep extraction (expert weight downsamples, MoA per-head attention maps, per-token expert
outputs): domain specialization only needs router logits.

**FFN router only — the attention (MoA) router is deliberately not recorded.** JetMoE's
defining feature is that attention is *also* Mixture-of-Experts — see
`docs/model-architecture-jetmoe-deepseek-research.md` §2 — and every layer still runs an MoA
router during these forward passes. This notebook simply leaves it unhooked. The Domain
Specialization tab visualizes FFN expert routing, which is also what the OLMoE schema
describes; an earlier version of this notebook emitted MoA aggregates alongside it under a
top-level `attention_routing` key, and that was dropped, since a field nothing reads is a field
that silently drifts out of date. (Note that `attention_routing` in the *routing trace* files,
which the Model Architecture tab does read, is a different key in a different file produced by
`extract_routing_trace_jetmoe.ipynb` — untouched by this.) The FFN router is top-2 of 8.

**Where the routing numbers come from.** Each FFN router module is hooked directly — a
post-hook takes the logits it returned (`JetMoeTopKGating` hands back the full 8-expert vector
at index 4), and a pre-hook takes the input it was called with, used to recompute those logits
independently as a cross-check. The statistics are built from the router's **own** logits: a
fp32 recompute from bf16 inputs lands ~0.4% away per logit, which is enough to flip the
2nd-vs-3rd expert on a flat 8-way router. Measured on the real model, that disagreed with the
FFN router's actual choice on ~2% of tokens (and on ~15% at attention-router layer 0, which is
how the effect was spotted in the first place). Those tokens are near-ties either way, but only
one of the two answers is what actually fired.

**Do not reconcile against `model(..., output_router_logits=True)`.** transformers records that
field from **both** routers — `OutputRecorder(JetMoeAttention, index=2)` *and*
`OutputRecorder(JetMoeTopKGating, index=4)` — so the returned list interleaves MoA and FFN logits
with no layout guarantee, and comparing our FFN layer *i* against its element *i* reports a huge
mismatch (~7.5) on a perfectly correct extraction. **This is still true now that the notebook
records the FFN router only**: the model has not changed, every layer still has an MoA router,
and that field is still a mix of the two. Hooking `mlp.router` directly, as below, is the only
safe path — do not "simplify" it away.

Run on a Colab A100 GPU runtime (8B params, ~16GB in bf16). 6 forward passes total.

**Expected output size ~2-3 MB**, dominated by `expert_token_idx` (24 layers x ~400 tokens x
top-2 x 6 domains ≈ 115k `[token_idx, score]` pairs). Comfortably smaller than OLMoE's 8.4 MB —
top-2 of 8 records far fewer pairs than top-8 of 64 — so it needs none of the per-prompt
splitting the routing traces required.

In [2]:
import importlib.util
import subprocess
import sys


def pip_install(*packages):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


if importlib.util.find_spec("torch") is None:
    pip_install("torch")

pip_install("transformers", "accelerate", "umap-learn", "numpy", "scikit-learn")

# JetMoE landed in mainline transformers very recently (JetMoeConfig / JetMoeForCausalLM,
# using Python 3.10+ union-type syntax and huggingface_hub's @strict config validation) --
# if the pip-published version is too old to have it, fall back to installing from source.
try:
    from transformers import JetMoeConfig  # noqa: F401
except ImportError:
    print("JetMoeConfig not found in installed transformers -- installing from GitHub main.")
    pip_install("git+https://github.com/huggingface/transformers.git")

print("Dependency installation complete.")

Dependency installation complete.


In [3]:
import json
import os

import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer, JetMoeConfig

MODEL_ID = "jetmoe/jetmoe-8b"
OUT_PATH = "jetmoe_domain_specialization.json"
UMAP_OUT_PATH = "jetmoe_domain_specialization_umap.json"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# The HF repo's config.json predates JetMoE's mainline transformers integration and still
# uses the model's original field names (num_layers, moe_num_experts, moe_top_k) instead of
# the current JetMoeConfig field names (num_hidden_layers, num_local_experts,
# num_experts_per_tok) -- see docs/model-architecture-jetmoe-deepseek-research.md §2.1's
# "Important config.json caveat". Verify against the paper's known-correct numbers and patch
# if the loader didn't pick them up, rather than silently building the wrong-sized model.
# Identical to extract_routing_trace_jetmoe.ipynb -- same model, same load path.
# NOTE: num_attention_heads is deliberately NOT in EXPECTED -- this dict is splatted into
# JetMoeConfig(...) below and __post_init__ derives num_attention_heads itself; it gets its own
# check right after the load.
EXPECTED = {
    "num_hidden_layers": 24,
    "num_local_experts": 8,
    "num_experts_per_tok": 2,
    "hidden_size": 2048,
    "num_key_value_heads": 16,
    "kv_channels": 128,
}

try:
    config = AutoConfig.from_pretrained(MODEL_ID)
except Exception as e:  # noqa: BLE001 -- deliberately broad: @strict's exact error type for
    # legacy/unrecognized config keys isn't confirmed, so catch anything and fall back rather
    # than let one specific exception class slip through uncaught.
    print(f"AutoConfig.from_pretrained raised {e!r} (likely the legacy-key mismatch above). "
          f"Falling back to an explicitly-constructed JetMoeConfig with known-correct values.")
    config = JetMoeConfig(**EXPECTED, vocab_size=32000, max_position_embeddings=4096, intermediate_size=5632)

mismatches = {k: (getattr(config, k, "<missing>"), v) for k, v in EXPECTED.items() if getattr(config, k, None) != v}
if mismatches:
    print(f"Config field mismatch after load: {mismatches}")
    print("Patching to known-correct values from the paper/config.json before loading weights.")
    for k, v in EXPECTED.items():
        setattr(config, k, v)

for k, v in EXPECTED.items():
    assert getattr(config, k) == v, f"config.{k} = {getattr(config, k)}, expected {v} -- investigate before proceeding"

# ---- GQA check: 32 query heads over 16 shared K/V heads ----
# Same block as extract_routing_trace_jetmoe.ipynb, and it has to run UNCONDITIONALLY (not only
# when `mismatches` is non-empty) and BEFORE from_pretrained -- patching a config the model was
# already built from does nothing.
#
# JetMoeConfig.__post_init__ derives num_attention_heads = num_key_value_heads * num_experts_per_tok
# = 16 * 2 = 32. That is not "16 heads": each of the top-2 selected attention experts contributes
# its own num_key_value_heads (16) query heads, and BOTH experts read the SAME 16 K/V heads --
# modeling_jetmoe.py stacks the experts' query heads on the head axis and does
# `key_states.repeat(1, top_k, 1, 1)` on the shared K/V. So the model runs 32 query heads against
# 16 K/V heads: grouped-query attention, 2 query heads per K/V head (paper Table 1 / research doc §2.2).
#
# This notebook records the FFN router only, so it never indexes an attention head -- but the
# attention (incl. its MoA router) still runs on every forward pass below, and it has to run with
# the architecture the checkpoint was trained with, or the hidden states feeding the FFN router
# are not the model's.
expected_query_heads = config.num_key_value_heads * config.num_experts_per_tok
if getattr(config, "num_attention_heads", None) != expected_query_heads:
    print(f"config.num_attention_heads = {getattr(config, 'num_attention_heads', '<missing>')}, "
          f"patching to {expected_query_heads} (num_key_value_heads x num_experts_per_tok, per JetMoeConfig.__post_init__).")
    config.num_attention_heads = expected_query_heads
assert config.num_attention_heads == 32 and config.num_key_value_heads == 16, (
    f"expected GQA 32 query heads / 16 K/V heads, got {config.num_attention_heads}/{config.num_key_value_heads}"
)

print("JetMoe config verified:", {k: getattr(config, k) for k in EXPECTED})
print(f"Attention is GQA: {config.num_attention_heads} query heads / {config.num_key_value_heads} K/V heads "
      f"({config.num_attention_heads // config.num_key_value_heads} query heads per K/V head), "
      f"head_dim {config.kv_channels}")

# No attn_implementation="eager" needed here -- this notebook never extracts attention weights,
# only FFN router logits, so the default attention kernel is fine (same reasoning as the OLMoE
# domain notebook). Attention still runs, of course -- including its own MoA router on every
# layer; that router is simply never hooked (see the header).
model, loading_info = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    config=config,
    dtype=torch.bfloat16,
    device_map="auto",
    output_loading_info=True,
)
model.eval()

assert not loading_info["missing_keys"], (
    f"Some model weights were NOT loaded from the checkpoint (randomly initialized "
    f"instead): {loading_info['missing_keys']}"
)
print(f"unexpected_keys (informational): {loading_info.get('unexpected_keys', [])}")

num_layers = config.num_hidden_layers
hidden_size = config.hidden_size
num_experts = config.num_local_experts        # JetMoeMoE.__init__ reads config.num_local_experts
top_k_experts = config.num_experts_per_tok    # JetMoeMoE.__init__ reads config.num_experts_per_tok
num_kv_heads = config.num_key_value_heads     # 16 shared K/V heads
num_query_heads = config.num_attention_heads  # 32 query heads per token, model-wide
head_dim = config.kv_channels                 # 128

# Fail fast if the FFN router module path this notebook depends on differs from the modeling source.
_l0 = model.model.layers[0]
assert hasattr(_l0.mlp, "router") and hasattr(_l0.mlp.router, "layer"), \
    f"expected mlp.router.layer.weight; mlp attrs={list(dict(_l0.mlp.named_children()))}"

# ---- structural proof of the 16 shared K/V heads (checkpoint weight shape, not just a config
# number): the ONE kv_proj per layer emits K and V for num_kv_heads heads of head_dim each, and
# every attention expert's query heads read those same heads. The per-expert Q/O weights are not
# checked here -- this notebook never reads them (that belongs in extract_routing_trace_jetmoe).
_kv_shape = tuple(_l0.self_attention.kv_proj.weight.shape)
assert _kv_shape == (2 * num_kv_heads * head_dim, hidden_size), (
    f"shared kv_proj is {_kv_shape}, expected {(2 * num_kv_heads * head_dim, hidden_size)} "
    f"(K and V, {num_kv_heads} heads x {head_dim} each)"
)

print(f"Loaded {MODEL_ID}: {num_layers} layers, FFN top-{top_k_experts} of {num_experts}; "
      f"attention GQA {num_query_heads} query heads over {num_kv_heads} shared K/V heads x {head_dim} dim "
      f"(kv_proj.weight={_kv_shape})")

config.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/968 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/266 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

unexpected_keys (informational): set()
Loaded jetmoe/jetmoe-8b: 24 layers, FFN top-2 of 8


In [4]:
# 6 domains x 1 long, coherent, grammatical passage each. All 6 passages were written from
# scratch for stylistic consistency (length, structure) -- including code/legal/biomedical,
# not just the 2 domains that are entirely new (math, conversational). The old "poetry"
# domain is retired in favor of "creative_writing". One prompt per domain keeps this to 6
# forward passes total; loading the model once is the fixed cost, prompt length barely
# affects memory, so longer passages here capture much richer per-domain routing signal.
DOMAIN_PROMPTS = {
    "code": [
        "Most production codebases begin with an interface contract before a single line of "
        "business logic gets written. A REST API endpoint is typically documented first with "
        "its request shape, response shape, and error codes, so that frontend and backend "
        "teams can work in parallel against a shared expectation rather than waiting on each "
        "other. Once the contract is settled, the backend team implements a service layer "
        "that validates input, applies business rules, and delegates persistence to a "
        "repository layer, keeping raw database queries out of the request handlers "
        "entirely.\n\n"
        "Consider a function that searches a sorted array for a target value using binary "
        "search. The function compares the target against the middle element, and if they "
        "do not match, discards the half of the array that cannot contain the target, "
        "repeating the process on the remaining half. This halves the search space on every "
        "comparison, giving binary search a logarithmic time complexity of O(log n), a "
        "dramatic improvement over the O(n) cost of scanning the array element by element, "
        "especially once the array grows into the millions of entries.\n\n"
        "Concurrency introduces its own category of bugs that rarely show up in "
        "single-threaded testing. A race condition occurs when two threads read and write "
        "shared state without proper synchronization, producing a result that depends on "
        "unpredictable timing rather than program logic. Developers guard against this with "
        "locks, atomic operations, or by redesigning the system around immutable data "
        "structures and message passing, so that no two threads ever mutate the same memory "
        "at the same time.\n\n"
        "Once a feature is implemented, it still has to survive code review and continuous "
        "integration before merging. A pull request typically triggers an automated pipeline "
        "that runs the unit test suite, checks code coverage, and lints the diff for style "
        "violations, failing the build before a human reviewer even looks at it if any of "
        "those checks do not pass. Only after the pipeline is green and a colleague has "
        "approved the change does it get merged into the main branch and queued for the "
        "next deployment."
    ],
    "math": [
        "Algebra gives us a systematic way to find unknown quantities from known "
        "relationships. To solve the equation 3x plus 7 equals 22, we isolate x by "
        "subtracting 7 from both sides to get 3x equals 15, then dividing both sides by 3 to "
        "find that x equals 5. This same principle of performing identical operations on "
        "both sides of an equation scales up to systems of many variables, which is the "
        "foundation of linear algebra and, eventually, of the matrix operations that power "
        "modern machine learning models.\n\n"
        "Calculus formalizes the idea of instantaneous change. The derivative of a function "
        "at a point measures the slope of the tangent line there, so the derivative of x "
        "cubed plus 2x with respect to x is 3x squared plus 2, telling us exactly how fast "
        "the function's output grows as x increases. Integration reverses this process, "
        "accumulating infinitely many infinitesimal slices to compute a total, such as the "
        "area under a curve or the distance traveled by an object whose velocity changes "
        "continuously over time.\n\n"
        "Geometry and number theory each contribute their own foundational facts. The "
        "Pythagorean theorem states that in a right triangle, the square of the hypotenuse "
        "equals the sum of the squares of the other two sides, a relationship that underlies "
        "everything from architectural design to GPS trilateration. A prime number, "
        "meanwhile, is a whole number greater than one that is divisible only by itself and "
        "one; the fact that every integer factors uniquely into primes is the basis of much "
        "of modern cryptography.\n\n"
        "Probability quantifies uncertainty using precise rules rather than intuition alone. "
        "If a fair six-sided die is rolled twice, the chance of rolling a six both times is "
        "one-sixth multiplied by one-sixth, or one in thirty-six, because the two rolls are "
        "independent events. This same multiplication rule, extended across thousands of "
        "variables, is what allows statisticians to model everything from election outcomes "
        "to the reliability of a manufactured part over its expected lifetime."
    ],
    "biomedical": [
        "A 58-year-old woman arrived at the emergency department reporting sudden, crushing "
        "chest pain that radiated into her jaw, along with nausea and cold sweats. An "
        "electrocardiogram showed ST-segment elevation in the anterior leads, consistent "
        "with an acute myocardial infarction, and she was taken directly to the "
        "catheterization lab, where an interventional cardiologist located and cleared a "
        "blockage in the left anterior descending artery. Within an hour of the blocked "
        "vessel being reopened, her chest pain had resolved and her cardiac enzyme levels "
        "began trending back toward normal.\n\n"
        "In a separate randomized, double-blind trial, researchers compared a new "
        "anti-inflammatory therapy against a placebo in patients with a chronic autoimmune "
        "condition. Participants who received the active treatment showed a statistically "
        "significant reduction in joint swelling and reported less pain on standardized "
        "questionnaires after twelve weeks, though a minority experienced mild "
        "injection-site irritation. The investigators concluded that the therapy was both "
        "effective and well tolerated, though they recommended a larger, multi-site "
        "follow-up trial before it could be considered for regulatory approval.\n\n"
        "At the molecular level, many of these therapies work by binding to a specific "
        "cell-surface receptor and blocking a signaling cascade that would otherwise trigger "
        "inflammation. This interrupts the release of pro-inflammatory cytokines, small "
        "proteins that normally recruit additional immune cells to a site of injury or "
        "infection, dampening the immune response without shutting it down entirely. A "
        "tissue biopsy taken before and after treatment can confirm this mechanism directly, "
        "typically showing reduced immune cell infiltration and lower levels of inflammatory "
        "markers such as C-reactive protein in the blood.\n\n"
        "Preventive medicine remains one of the most cost-effective tools available to "
        "clinicians. Routine vaccination trains the immune system to recognize a pathogen's "
        "distinctive surface proteins well before a real infection occurs, so that "
        "antibodies and memory immune cells are already circulating by the time exposure "
        "happens. Regular screening, similarly, catches conditions like hypertension or "
        "early-stage cancer while they are still asymptomatic and far easier to treat, often "
        "years before they would otherwise have produced any noticeable symptoms."
    ],
    "legal": [
        "This Master Services Agreement is entered into between the Client and the Service "
        "Provider as of the Effective Date, and governs all statements of work executed "
        "under it. The Service Provider agrees to deliver the services described in each "
        "statement of work in a professional and workmanlike manner, and the Client agrees "
        "to pay all undisputed invoices within thirty days of receipt. Either party may "
        "terminate the Agreement for convenience upon sixty days' written notice, provided "
        "that any fees accrued for work performed prior to the termination date remain due "
        "and payable in full.\n\n"
        "In a subsequent dispute, the plaintiff alleged that the defendant had breached a "
        "supply agreement by failing to deliver conforming goods by the contractually "
        "specified deadline. At trial, the court heard testimony from an industry expert "
        "regarding customary delivery timelines, together with internal correspondence in "
        "which the defendant acknowledged awareness of the deadline and its likely inability "
        "to meet it. The jury found that the defendant's failure to perform was a material "
        "breach and awarded damages calculated to place the plaintiff in the position it "
        "would have occupied had the contract been properly performed.\n\n"
        "On appeal, the defendant argued that the trial court's jury instructions on "
        "materiality were legally deficient and warranted a new trial. The appellate panel "
        "disagreed, holding that the instructions, considered in their entirety, correctly "
        "stated the governing legal standard, and that any imprecision in a single sentence "
        "did not amount to reversible error given the overwhelming weight of the evidence "
        "presented. The panel further reaffirmed that in civil actions the burden rests on "
        "the plaintiff to establish each element of the claim by a preponderance of the "
        "evidence, a standard it found comfortably satisfied on this record.\n\n"
        "Beyond contract and tort claims, corporate counsel also spend considerable time on "
        "regulatory compliance. Before launching a new product in a foreign jurisdiction, a "
        "company typically commissions a legal opinion addressing local licensing "
        "requirements, data protection obligations, and any sector-specific restrictions that "
        "might apply, since noncompliance can expose the company to fines, injunctions, or "
        "the forced withdrawal of the product from that market entirely."
    ],
    "creative_writing": [
        "The lighthouse keeper had watched a thousand storms roll in off the grey Atlantic, "
        "but something about this one made him pause at the window with his tea going cold "
        "in his hand. The waves were climbing higher than the rocks that had stood against "
        "them for three hundred years, and for the first time in his forty seasons on the "
        "island, he found himself counting the ships he could see and hoping the count would "
        "not change by morning.\n\n"
        "Mira found the letter tucked inside a hollowed-out book on her grandmother's shelf, "
        "the paper gone soft and yellow at the folds. Her hands trembled as she unfolded it, "
        "not from cold but from the particular fear of learning something that could not be "
        "unlearned, and when she finally read the first line, she understood at once why it "
        "had been hidden rather than simply thrown away.\n\n"
        "Deep in the forest, where the canopy grew so thick that noon light arrived the color "
        "of dusk, the old paths remembered every traveler who had ever walked them. The wind "
        "moved through the high branches in long, unhurried sighs, and if you stood still "
        "long enough and let your own breathing slow to match it, you could almost believe "
        "the trees were arguing quietly among themselves about whether to let you pass.\n\n"
        "By the time the last streetlamp flickered out, the city had already begun its other "
        "life, the one that belonged to the people who swept its floors and stocked its "
        "shelves while everyone else slept. A fox slipped across the empty intersection "
        "without breaking stride, entirely unbothered by the traffic lights still cycling to "
        "no one, and somewhere above the rooftops the sky was already deciding, slowly, what "
        "color the morning would be."
    ],
    "conversational": [
        "Hey, sorry for the late reply, my phone died on the way home and I didn't get a "
        "chance to charge it until just now. Anyway, are we still good for Saturday, or did "
        "something come up on your end? I can also do Sunday afternoon if that works better, "
        "just let me know so I can figure out the rest of my weekend around it.\n\n"
        "Honestly, I've been kind of exhausted this week, nothing serious, just one of those "
        "stretches where every day feels a little longer than it should. I think I just need "
        "a weekend where I don't have anywhere to be, maybe cook something simple, watch a "
        "movie I've already seen a dozen times, that kind of thing. How about you, anything "
        "fun happen lately, or has it been the same kind of week over there?\n\n"
        "Oh, that reminds me, did you end up trying that new place downtown? A couple of "
        "people at work were talking about it and apparently the line gets pretty long on "
        "weekends, so if we want to check it out we should probably go early or just do a "
        "weekday evening instead. I'm not picky either way, honestly whatever's easiest works "
        "for me, I just haven't had a good excuse to get out of the house in a while.\n\n"
        "Thanks again for helping me move that bookshelf last week, by the way, I really owe "
        "you one. Let me know if you ever need a hand with anything, moving, fixing something "
        "around the house, whatever, I'm around most weekends these days. Talk soon, and "
        "text me whenever about Saturday, no rush."
    ],
}

domains = list(DOMAIN_PROMPTS.keys())
print(f"Domains: {domains}")
for domain, prompts in DOMAIN_PROMPTS.items():
    assert len(prompts) == 1, f"{domain} has {len(prompts)} prompts, expected 1"
    print(f"  {domain}: {len(prompts[0].split())} words")


Domains: ['code', 'math', 'biomedical', 'legal', 'creative_writing', 'conversational']
  code: 341 words
  math: 326 words
  biomedical: 334 words
  legal: 349 words
  creative_writing: 293 words
  conversational: 271 words


In [5]:
# The statistics are built from the FFN router's OWN logits, taken straight off a post-hook on
# mlp.router: JetMoeTopKGating.forward returns
# (index_sorted_experts, batch_index, batch_gates, expert_size, logits) and element 4 is the full
# pre-softmax vector over all 8 experts -- everything this notebook needs. (JetMoE differs from
# DeepSeek here: DeepSeek's MoEGate returns only top-k bookkeeping, so that notebook has no choice
# but to recompute.) We still apply our own FULL softmax over all 8, because the gating itself
# softmaxes over the top-2 only, and avg_prob counts every expert, selected or not.
#
# Using the model's own logits matters, not just for elegance: a fp32 recompute from bf16 inputs
# lands ~0.4% away per logit, which is enough to flip the 2nd-vs-3rd expert on a flat 8-way
# router. Measured on the real model, that disagreed with the FFN router's actual choice on ~2%
# of tokens. Those tokens are near-ties, so either answer is defensible -- but only one of them is
# what actually fired, and that is the one worth counting.
#
# A pre-hook on the same module captures the input, used to recompute the logits independently as
# a cross-check (below): it validates that this really is the router and that its weight path is
# what we think, and it is the fallback if a future version reorders the gating's return.
#
# Only the FFN router is hooked. Attention still runs its own MoA router on every layer, but this
# notebook deliberately does not record it -- see the header.
#
# Do NOT reconcile against model(..., output_router_logits=True): transformers records that field
# from BOTH JetMoeAttention (index 2) and JetMoeTopKGating (index 4), so the returned list mixes
# the MoA and FFN routers with no layout guarantee, and comparing our FFN layer li against its
# element li is apples-to-oranges (it reports diffs of ~7.5 on a correct extraction). That remains
# true even though nothing below touches MoA: the model still has those routers.
def stats_for_prompt(prompt, check_against_model=False):
    """Per-layer [num_experts] FFN top-k hit counts and summed probs, plus token count, each
    token's decoded text, and per (layer, expert) the (token index, routing score) pairs that
    actually selected it -- the score lets callers rank tokens by how strongly they activated the
    expert, not just occurrence order.

    Returns (hit_counts, prob_sums, expert_token_idx, n_tokens, token_strs, reconcile) where
    reconcile is None unless check_against_model=True, in which case it is a list of
    (layer, logit_rel_diff, selection_agreement, tie_fraction) comparing the independent recompute
    against the router's own logits -- see the smoke cell for how to read them."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    token_ids = inputs["input_ids"][0].tolist()
    token_strs = [tokenizer.decode([tid]) for tid in token_ids]
    n_tokens = len(token_ids)

    router_inputs = {}
    router_returns = {}
    hooks = []

    # Both hooks append rather than assign: this extraction is only valid if the router makes
    # exactly one routing decision per token per layer, so the call count is asserted below
    # rather than assumed.
    def make_pre_hook(store, li):
        def hook(module, args):
            store.setdefault(li, []).append(args[0])
        return hook

    def make_post_hook(store, li):
        def hook(module, args, output):
            store.setdefault(li, []).append(output)
        return hook

    for li, layer in enumerate(model.model.layers):
        router = layer.mlp.router
        hooks.append(router.register_forward_pre_hook(make_pre_hook(router_inputs, li)))
        hooks.append(router.register_forward_hook(make_post_hook(router_returns, li)))

    with torch.no_grad():
        model(**inputs)

    for h in hooks:
        h.remove()

    def as_tokens_by_hidden(t):
        """The router is called with [bs*seq, hidden] (JetMoeMoE reshapes before calling), but
        accept [1, seq, hidden] too so this doesn't break if that ever changes."""
        x = t.detach().float().cpu()
        if x.dim() == 3:
            x = x[0]
        assert x.shape[0] == n_tokens, f"router input has {x.shape[0]} rows, expected {n_tokens}"
        return x

    def gating_logits(returned, n_exp):
        """JetMoeTopKGating.forward returns
        (index_sorted_experts, batch_index, batch_gates, expert_size, logits) -- element 4. Scan
        for a [n_tokens, n_exp] float tensor rather than trusting the position blindly, so a
        reordered return in some future version degrades to 'not comparable', not to a wrong
        comparison."""
        items = list(returned) if isinstance(returned, (tuple, list)) else [returned]
        for cand in (items[4:5] + items):  # documented position first, then any match
            if torch.is_tensor(cand) and cand.is_floating_point() and cand.dim() == 2 \
                    and cand.shape == (n_tokens, n_exp):
                return cand.detach().float().cpu()
        return None

    reconcile = [] if check_against_model else None
    fallbacks = []
    assert set(router_inputs) == set(range(num_layers)), (
        f"FFN router pre-hooks fired for layers {sorted(router_inputs)}, "
        f"expected all {num_layers}"
    )
    hit_counts = torch.zeros(num_layers, num_experts)
    prob_sums = torch.zeros(num_layers, num_experts)
    expert_token_idx = [[[] for _ in range(num_experts)] for _ in range(num_layers)]
    for li in range(num_layers):
        calls = router_inputs[li]
        assert len(calls) == 1, (
            f"FFN router at layer {li} ran {len(calls)} times in one forward -- this "
            f"extraction assumes one routing decision per token per layer"
        )
        h = as_tokens_by_hidden(calls[0])
        weight = model.model.layers[li].mlp.router.layer.weight.detach().float().cpu()
        ours = F.linear(h, weight)  # independent recompute, fp32 -- cross-check only
        assert ours.shape == (n_tokens, num_experts), f"layer {li}: got {tuple(ours.shape)}"

        # What the router itself produced. This is what the model actually routed on, so it
        # is what the statistics count; `ours` only ever validates it.
        theirs = gating_logits((router_returns.get(li) or [None])[0], num_experts)
        if theirs is None:
            fallbacks.append(li)
            logits = ours
        else:
            logits = theirs

        probs = torch.softmax(logits, dim=-1)  # full softmax over all experts, our convention
        topk = torch.topk(probs, k=top_k_experts, dim=-1).indices
        for t in range(n_tokens):
            hit_counts[li, topk[t]] += 1
            for e in topk[t].tolist():
                expert_token_idx[li][e].append((t, round(float(probs[t, e]), 5)))
        prob_sums[li] += probs.sum(dim=0)

        if check_against_model and theirs is not None:
            rel = ((theirs - ours).abs().max() / theirs.abs().max().clamp(min=1e-6)).item()
            our_topk = torch.topk(ours, k=top_k_experts, dim=-1).indices
            agree = (torch.sort(our_topk, dim=-1).values == torch.sort(topk, dim=-1).values) \
                .all(dim=-1).float().mean().item()
            # How many tokens COULD flip: those whose k-th vs (k+1)-th logit gap is within the
            # observed noise. A gap is a difference of two logits, so the noise on it is up to
            # 2x the per-logit noise. Any disagreement above this fraction would not be
            # explainable by rounding, and would mean a genuinely wrong input or weight.
            noise = 2.0 * (theirs - ours).abs().max()
            ordered = theirs.sort(dim=-1, descending=True).values
            gap = ordered[:, top_k_experts - 1] - ordered[:, top_k_experts]
            tie_frac = (gap <= noise).float().mean().item()
            reconcile.append((li, rel, agree, tie_frac))

    if fallbacks:
        print(f"    [warn] the gating return had no usable logits for layers {fallbacks[:5]} "
              f"({len(fallbacks)} total) -- those fell back to the fp32 recompute, which can "
              f"differ from the model's own choice on near-tied tokens")

    return hit_counts, prob_sums, expert_token_idx, n_tokens, token_strs, reconcile

In [6]:
# Smoke pass on the shortest domain before the full sweep: confirms that the FFN router's
# recomputed logits reproduce that router's own output (module-level ground truth, every layer),
# and that the routing is structurally sane -- exactly top-k experts per token, a full softmax
# over all 8, and not collapsed onto a single expert (which would mean the wrong hidden state is
# feeding the router).
_smoke_domain = min(DOMAIN_PROMPTS, key=lambda d: len(DOMAIN_PROMPTS[d][0]))
print(f"smoke domain: {_smoke_domain}")
_hits, _probs, _e_idx, _n_tok, _toks, _reconcile = stats_for_prompt(
    DOMAIN_PROMPTS[_smoke_domain][0], check_against_model=True)

print(f"tokens: {_n_tok}; first 8: {_toks[:8]}")
print(f"hits per token per layer: {(_hits.sum(dim=1) / _n_tok).tolist()[:4]} ... (expect {top_k_experts} everywhere)")
assert torch.allclose(_hits.sum(dim=1), torch.full((num_layers,), float(top_k_experts * _n_tok))), \
    "FFN hit counts do not sum to top_k * n_tokens per layer -- top-k extraction is wrong"
assert torch.allclose(_probs.sum(dim=1), torch.full((num_layers,), float(_n_tok)), atol=1e-2), \
    "FFN per-layer probabilities do not sum to n_tokens -- softmax is over the wrong axis"

_rate = (_hits / _n_tok)
print(f"layer 0 FFN activation rates: {[round(v, 3) for v in _rate[0].tolist()]}")
print(f"max single-expert rate across layers: {_rate.max().item():.3f} "
      f"(1.0 would mean one expert takes every token -- possible for top-2 of 8, but not everywhere)")
assert (_rate.max(dim=1).values < 0.999).any(), \
    "every layer routes ALL tokens to one expert -- the router input is almost certainly wrong"

# Cross-check, per layer: an independent fp32 recompute (the router's own hooked input through
# its own weight) against the logits the router itself returned -- which is what the statistics
# above are actually built from.
#
# The meaningful gate is the RELATIVE LOGIT DIFF. If the hooked input or the weight path were
# wrong, it would be enormous (comparing against the wrong router reports ~1.5 here).
#
# Top-k AGREEMENT is deliberately NOT gated on a round number. The model computes its logits in
# bf16 and this recompute runs in fp32, so on a flat router the 2nd-vs-3rd expert can genuinely
# swap. What must hold is that every disagreement is explainable by a near-tie: the fraction of
# tokens whose k-th vs (k+1)-th gap sits within the observed noise is an upper bound on how many
# could flip, so disagreement must not exceed it. That is a real test of the extraction; a
# ">0.95" threshold would only be a number that happened to pass -- do not reintroduce one.
assert _reconcile, "reconcile list is empty -- the post-hooks captured nothing"

TIE_SLACK = 0.02  # tolerance on the "explainable by a near-tie" bound

_worst_rel_layer, _worst_rel = max(((li, rel) for li, rel, _, _ in _reconcile), key=lambda p: p[1])
_worst_agr_layer, _worst_agr = min(((li, a) for li, _, a, _ in _reconcile), key=lambda p: p[1])
# The layer where disagreement comes closest to exceeding what ties can explain.
_tightest_layer, _tightest = max(((li, (1 - a) - tie) for li, _, a, tie in _reconcile),
                                 key=lambda p: p[1])
print(f"[cross-check] FFN router: {len(_reconcile)}/{num_layers} layers | "
      f"relative logit diff worst {_worst_rel:.5f} (layer {_worst_rel_layer}) | "
      f"top-k agreement worst {_worst_agr:.4f} (layer {_worst_agr_layer}) | "
      f"unexplained-by-ties worst {_tightest:+.4f} (layer {_tightest_layer}, "
      f"must stay <= {TIE_SLACK})")
assert len(_reconcile) == num_layers, (
    f"only {len(_reconcile)}/{num_layers} layers could be compared -- see the messages above"
)
assert _worst_rel < 0.05, (
    f"layer {_worst_rel_layer} logits differ from the router's own by {_worst_rel:.4f} relative "
    f"-- far more than bf16-vs-fp32 rounding, so the hooked input or the weight path is wrong. "
    f"Fix before sweeping."
)
assert _tightest <= TIE_SLACK, (
    f"layer {_tightest_layer}: top-k disagreement exceeds what near-ties can explain by "
    f"{_tightest:.4f}. Rounding alone cannot account for this -- investigate before sweeping."
)

print("The statistics above use the router's OWN logits; the recompute is only a cross-check.")
print("Smoke test passed.")

smoke domain: conversational
tokens: 341; first 8: ['<s>', 'Hey', ',', 'sorry', 'for', 'the', 'late', 'reply']
hits per token per layer: [2.0, 2.0, 2.0, 2.0] ... (expect 2 everywhere)
layer 0 FFN activation rates: [0.211, 0.311, 0.214, 0.261, 0.226, 0.276, 0.284, 0.217]
max single-expert rate across layers: 0.460 (1.0 would mean one expert takes every token -- possible for top-2 of 8, but not everywhere)
[cross-check] FFN router: 24/24 layers | relative logit diff worst 0.00370 (layer 22) | top-k agreement worst 0.9824 (layer 17) | unexplained-by-ties worst -0.0147 (layer 4, must stay <= 0.02)
The statistics above use the router's OWN logits; the recompute is only a cross-check.
Smoke test passed.


In [7]:
activation_rate = {}    # domain -> [layer][expert]  fraction of domain's tokens with expert in top-k
avg_prob = {}           # domain -> [layer][expert]  mean router softmax prob (selected or not)
token_counts = {}
prompt_counts = {}
# domain -> [token_str, ...] and domain -> [layer][expert] -> [(token_idx, score), ...] into
# that list, so the popup can show exactly which real tokens routed to a given expert/layer.
domain_tokens = {}
expert_token_idx = {}

for domain, prompts in DOMAIN_PROMPTS.items():
    print(f"\n== domain: {domain} ==")
    assert len(prompts) == 1, "domain_tokens/expert_token_idx assume exactly one prompt per domain"
    prompt = prompts[0]
    print(f"  {prompt[:80]!r}...")
    hits, probs, e_idx, n_tok, token_strs, _ = stats_for_prompt(prompt)

    activation_rate[domain] = (hits / n_tok).tolist()
    avg_prob[domain] = (probs / n_tok).tolist()
    token_counts[domain] = n_tok
    prompt_counts[domain] = len(prompts)
    domain_tokens[domain] = token_strs
    expert_token_idx[domain] = e_idx

    print(f"  total tokens: {n_tok}")


== domain: code ==
  'Most production codebases begin with an interface contract before a single line '...
  total tokens: 424

== domain: math ==
  'Algebra gives us a systematic way to find unknown quantities from known relation'...
  total tokens: 442

== domain: biomedical ==
  'A 58-year-old woman arrived at the emergency department reporting sudden, crushi'...
  total tokens: 492

== domain: legal ==
  'This Master Services Agreement is entered into between the Client and the Servic'...
  total tokens: 456

== domain: creative_writing ==
  'The lighthouse keeper had watched a thousand storms roll in off the grey Atlanti'...
  total tokens: 359

== domain: conversational ==
  "Hey, sorry for the late reply, my phone died on the way home and I didn't get a "...
  total tokens: 341


In [8]:
# Derived statistics, identical formulas to extract_domain_specialization.ipynb -- kept as a
# function taking (rates, n_exp, top_k) rather than reading the globals directly, so the expert
# count and top-k are never assumed.
EPS = 1e-4


def derive(activation_rate, n_exp, top_k):
    """Synthetic baseline: none of the 6 domains is meant to be neutral/generic text, so instead
    of a 7th hand-authored "baseline" passage, use the mean activation rate across the 6 domains
    themselves, per (layer, expert), as the reference point."""
    baseline_rate = [
        [sum(activation_rate[d][li][e] for d in domains) / len(domains) for e in range(n_exp)]
        for li in range(num_layers)
    ]

    # specialization_score[domain][layer][expert] = log2((rate_domain + eps) / (rate_baseline + eps))
    # -- positive = over-used relative to the 6-domain average.
    specialization_score = {
        d: [
            [round(float(np.log2((activation_rate[d][li][e] + EPS) / (baseline_rate[li][e] + EPS))), 4)
             for e in range(n_exp)]
            for li in range(num_layers)
        ]
        for d in domains
    }

    # layer_divergence[domain][layer] = total-variation distance between the domain's and the
    # synthetic baseline's per-expert selection distribution (each normalized to sum to 1 across
    # experts via /top_k). 0 = identical routing to the 6-domain average at that layer.
    layer_divergence = {}
    for d in domains:
        divs = []
        for li in range(num_layers):
            dom_dist = [activation_rate[d][li][e] / top_k for e in range(n_exp)]
            base_dist = [baseline_rate[li][e] / top_k for e in range(n_exp)]
            divs.append(round(0.5 * sum(abs(a - b) for a, b in zip(dom_dist, base_dist)), 5))
        layer_divergence[d] = divs

    # domain_rate[domain][layer] = mean activation_rate of that domain's top-K most-used experts
    # at that layer. NOTE the cross-model caveat: K is the model's own top-k (2 for JetMoE, 8 for
    # OLMoE, 6 for DeepSeek), so this number is comparable across domains WITHIN a model, but not
    # directly across models.
    domain_rate = {
        d: [round(sum(sorted(activation_rate[d][li], reverse=True)[:top_k]) / top_k, 5)
            for li in range(num_layers)]
        for d in domains
    }

    # top experts per domain, ranked directly by real activation_rate (no baseline comparison)
    top_specialists = {}
    for d in domains:
        pairs = [(activation_rate[d][li][e], li, e) for li in range(num_layers) for e in range(n_exp)]
        pairs.sort(reverse=True)
        top_specialists[d] = [
            {"layer": li, "expert": e, "activation_rate": round(rate, 4)} for rate, li, e in pairs[:12]
        ]

    return specialization_score, layer_divergence, domain_rate, top_specialists


specialization_score, layer_divergence, domain_rate, top_specialists = derive(
    activation_rate, num_experts, top_k_experts)

for d in domains:
    print(f"{d:>17}: FFN domain_rate L0={domain_rate[d][0]:.3f} L{num_layers - 1}={domain_rate[d][-1]:.3f} | "
          f"top expert {top_specialists[d][0]}")

             code: FFN domain_rate L0=0.297 L23=0.380 | top expert {'layer': 22, 'expert': 4, 'activation_rate': 0.566}
             math: FFN domain_rate L0=0.305 L23=0.345 | top expert {'layer': 8, 'expert': 7, 'activation_rate': 0.457}
       biomedical: FFN domain_rate L0=0.309 L23=0.311 | top expert {'layer': 18, 'expert': 1, 'activation_rate': 0.439}
            legal: FFN domain_rate L0=0.306 L23=0.339 | top expert {'layer': 7, 'expert': 7, 'activation_rate': 0.4079}
 creative_writing: FFN domain_rate L0=0.281 L23=0.380 | top expert {'layer': 11, 'expert': 7, 'activation_rate': 0.5237}
   conversational: FFN domain_rate L0=0.298 L23=0.389 | top expert {'layer': 10, 'expert': 1, 'activation_rate': 0.4604}


In [9]:
# Every field here describes the FFN router, exactly as the OLMoE schema does -- no
# JetMoE-specific keys, so the Domain Specialization tab needs no per-model special-casing.
out = {
    "domains": domains,
    "num_layers": num_layers,
    "num_experts": num_experts,
    "top_k_experts": top_k_experts,
    "token_counts": token_counts,
    "prompt_counts": prompt_counts,
    "example_prompts": DOMAIN_PROMPTS,
    "activation_rate": {d: [[round(v, 5) for v in row] for row in activation_rate[d]] for d in domains},
    "avg_prob": {d: [[round(v, 5) for v in row] for row in avg_prob[d]] for d in domains},
    "specialization_score": specialization_score,
    "layer_divergence": layer_divergence,
    "domain_rate": domain_rate,
    "domain_tokens": domain_tokens,
    "expert_token_idx": expert_token_idx,
    "top_specialists": top_specialists,
}

with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(out, f)

print(f"\nWrote domain specialization data to {OUT_PATH} ({os.path.getsize(OUT_PATH) / 1e3:.1f} KB)")


Wrote domain specialization data to jetmoe_domain_specialization.json (1959.3 KB)


## UMAP: (layer, expert) activation across domains

Reuses the `activation_rate` computed above (no extra forward passes) to build one vector
per (layer, expert) pair, one dimension per domain, and projects it to 2D with cosine-metric
UMAP -- same method as `extract_domain_specialization.ipynb`. All-zero (never-activated) pairs
are excluded from the projection and reported separately as `excluded_experts`.

FFN experts only, matching `jetmoe_routing_trace_umap.json` and everything above -- the
attention (MoA) router is not recorded by this notebook at all.

Note the scale: 24 layers x 8 experts = 192 points, versus OLMoE's 1,024. With only 8 experts
per layer and top-2 routing, expect a sparser, more banded map than OLMoE's.

In [10]:
import umap

NUM_LAYERS = num_layers
NUM_EXPERTS = num_experts

expert_vectors = np.array([
    [activation_rate[d][li][e] for d in domains]
    for li in range(NUM_LAYERS)
    for e in range(NUM_EXPERTS)
])

point_layer_ids = np.repeat(np.arange(NUM_LAYERS), NUM_EXPERTS)
point_expert_ids = np.tile(np.arange(NUM_EXPERTS), NUM_LAYERS)

# Never-activated (layer, expert) pairs are all-zero and undefined under the cosine metric --
# exclude from the projection, report separately as excluded_experts.
active_mask = expert_vectors.sum(axis=1) > 0
active_vectors = expert_vectors[active_mask]

# n_neighbors must stay below the point count; 192 points is well above 15, but guard anyway so
# this doesn't become a silent failure if the expert count ever changes.
n_neighbors = min(15, max(2, active_vectors.shape[0] - 1))
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=n_neighbors, min_dist=0.1,
                    metric="cosine", n_jobs=1)
active_embedding = reducer.fit_transform(active_vectors)

assert not np.isnan(active_embedding).any(), (
    "UMAP produced NaN coordinates even after excluding all-zero rows -- inspect "
    "active_vectors for degenerate rows, or re-run with metric='euclidean'."
)

print(f"Built {expert_vectors.shape[0]} (layer, expert) vectors across {len(domains)} domains.")
print(f"UMAP embedding shape: {active_embedding.shape} ({int(active_mask.sum())} active of {expert_vectors.shape[0]} total pairs)")

TOP_K_TOKENS = 8
active_indices = np.flatnonzero(active_mask)

umap_points = []
for row, i in enumerate(active_indices):
    layer_id = int(point_layer_ids[i])
    expert_id = int(point_expert_ids[i])
    vec = expert_vectors[i]
    dominant_domain = domains[int(np.argmax(vec))]

    # top_tokens: pool every (token, routing score) pair that selected this (layer, expert)
    # across all domains, then keep the TOP_K_TOKENS with the highest score -- so the hover
    # popup surfaces the tokens that activated this expert most strongly.
    pooled = [
        (score, domain_tokens[d][t_idx], d)
        for d in domains
        for t_idx, score in expert_token_idx[d][layer_id][expert_id]
    ]
    pooled.sort(key=lambda item: -item[0])
    top_tokens = [
        {"token": tok, "score": round(float(score), 4), "domain": d}
        for score, tok, d in pooled[:TOP_K_TOKENS]
    ]

    umap_points.append({
        "layer_id": layer_id,
        "expert_id": expert_id,
        "x": round(float(active_embedding[row, 0]), 4),
        "y": round(float(active_embedding[row, 1]), 4),
        "dominant_domain": dominant_domain,
        "domain_activation_rate": {d: round(float(vec[j]), 4) for j, d in enumerate(domains)},
        "top_tokens": top_tokens,
    })

excluded_experts = [
    {"layer_id": int(point_layer_ids[i]), "expert_id": int(point_expert_ids[i])}
    for i in np.flatnonzero(~active_mask)
]

assert len(umap_points) + len(excluded_experts) == NUM_LAYERS * NUM_EXPERTS

umap_data = {
    "domains": domains,
    "num_layers": NUM_LAYERS,
    "num_experts": NUM_EXPERTS,
    "points": umap_points,
    "excluded_experts": excluded_experts,
}

with open(UMAP_OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(umap_data, f, ensure_ascii=False, allow_nan=False, indent=2)

print(f"Wrote {UMAP_OUT_PATH} ({len(umap_points)} points, {len(excluded_experts)} excluded pairs)")

Built 192 (layer, expert) vectors across 6 domains.
UMAP embedding shape: (192, 2) (192 active of 192 total pairs)
Wrote jetmoe_domain_specialization_umap.json (192 points, 0 excluded pairs)


In [11]:
try:
    from google.colab import files
    files.download(OUT_PATH)
    files.download(UMAP_OUT_PATH)
except ImportError:
    print("Not running in Google Colab -- skipping auto-download.")
    print(f"Files were written locally at: {OUT_PATH} and {UMAP_OUT_PATH}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>